# Random Forest Classification Lab

**Aim:** Build a Random Forest Classifier on the Insurance dataset and compare different `max_depth` and `n_estimators` using Accuracy, Precision and Recall.

Target `high_charges`: 1 = charges above median, 0 = at/below median.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
df = pd.read_csv("insurance.csv")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
median_charges = df["charges"].median()
df["high_charges"] = (df["charges"] > median_charges).astype(int)
print("Median charges:", median_charges)
print(df["high_charges"].value_counts())

In [ ]:
data = df.copy()
le = LabelEncoder()
data["sex"] = le.fit_transform(data["sex"])
data["smoker"] = le.fit_transform(data["smoker"])
data["region"] = le.fit_transform(data["region"])

X = data.drop(["charges", "high_charges"], axis=1)
y = data["high_charges"]
print(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# Different tree heights
depths = [2, 4, 6, 8, 10]
depth_results = []

for depth in depths:
    model = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    depth_results.append({
        "Max Depth": depth,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    })

depth_results = pd.DataFrame(depth_results)
depth_results

In [ ]:
plt.figure(figsize=(8, 5))
for m in ["Accuracy", "Precision", "Recall"]:
    plt.plot(depth_results["Max Depth"], depth_results[m], marker="o", label=m)
plt.xlabel("Maximum Tree Depth"); plt.ylabel("Score")
plt.title("Random Forest Performance vs Tree Height")
plt.legend(); plt.grid(True); plt.show()

depth_results.set_index("Max Depth")[["Accuracy", "Precision", "Recall"]].plot(kind="bar", figsize=(9, 5))
plt.title("Accuracy, Precision and Recall for Different Tree Heights")
plt.xlabel("Maximum Tree Depth"); plt.ylabel("Score"); plt.ylim(0, 1)
plt.xticks(rotation=0); plt.grid(axis="y"); plt.show()

In [ ]:
# Different numbers of trees
tree_numbers = [10, 50, 100, 150, 200]
tree_results = []

for n in tree_numbers:
    model = RandomForestClassifier(n_estimators=n, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    tree_results.append({
        "Number of Trees": n,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    })

tree_results = pd.DataFrame(tree_results)
tree_results

In [ ]:
plt.figure(figsize=(8, 5))
for m in ["Accuracy", "Precision", "Recall"]:
    plt.plot(tree_results["Number of Trees"], tree_results[m], marker="o", label=m)
plt.xlabel("Number of Trees"); plt.ylabel("Score")
plt.title("Random Forest Performance vs Number of Trees")
plt.legend(); plt.grid(True); plt.show()

tree_results.set_index("Number of Trees")[["Accuracy", "Precision", "Recall"]].plot(kind="bar", figsize=(9, 5))
plt.title("Accuracy, Precision and Recall for Different Numbers of Trees")
plt.xlabel("Number of Trees"); plt.ylabel("Score"); plt.ylim(0, 1)
plt.xticks(rotation=0); plt.grid(axis="y"); plt.show()

In [ ]:
best_depth = depth_results.loc[depth_results["Accuracy"].idxmax()]
best_tree = tree_results.loc[tree_results["Accuracy"].idxmax()]
print("Best Tree Height Result:"); print(best_depth)
print("\nBest Number of Trees Result:"); print(best_tree)

In [ ]:
final_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)

print("Final Model Results")
print("-------------------")
print("Accuracy :", round(accuracy_score(y_test, final_pred), 4))
print("Precision:", round(precision_score(y_test, final_pred), 4))
print("Recall   :", round(recall_score(y_test, final_pred), 4))

## Conclusion
- Accuracy ≈ 0.93 across settings; best was depth 8 / 10 trees (0.9366).
- Deeper trees add flexibility but can overfit; more trees make the forest more stable but cost more compute.
- Final model (100 trees, depth 6): Accuracy 0.9328, Precision 0.9677, Recall 0.8955.